<a href="https://colab.research.google.com/github/mochamadx5/MachineLearning_11_MochamadReza/blob/main/JS09/JS09_TugasLab01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install library yang dibutuhkan jika belum terpasang
!pip install annoy faiss-cpu hnswlib tabulate

import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
import pandas as pd

# Pastikan reproducibility
np.random.seed(42)

def compute_recall(I_pred, I_gt, k=10):
    correct = 0
    for i in range(len(I_pred)):
        correct += len(set(I_pred[i][:k]) & set(I_gt[i][:k]))
    return correct / (len(I_pred) * k)

def benchmark_scenario(n_data, dim, metric="euclidean", k=10, nq=100):
    print(f"\n=======================================================")
    print(f"Menjalankan: Data={n_data}, Dim={dim}D, Metrik={metric.upper()}")
    print(f"=======================================================")

    # 1. Generate Dataset
    xb = np.random.random((n_data, dim)).astype('float32')
    xq = np.random.random((nq, dim)).astype('float32')

    if metric == "cosine":
        # Normalisasi vektor untuk metrik Cosine
        faiss.normalize_L2(xb)
        faiss.normalize_L2(xq)
        annoy_metric = 'angular'
        hnsw_metric = 'cosine'
        faiss_metric = faiss.METRIC_INNER_PRODUCT
    else:
        annoy_metric = 'euclidean'
        hnsw_metric = 'l2'
        faiss_metric = faiss.METRIC_L2

    # 2. Ground Truth (Exact Brute Force)
    if metric == "cosine":
        gt_index = faiss.IndexFlatIP(dim)
    else:
        gt_index = faiss.IndexFlatL2(dim)
    gt_index.add(xb)
    _, gt_idx = gt_index.search(xq, k)

    results = {}

    # ----------------------------------------------------
    # A. ANNOY
    # ----------------------------------------------------
    t0 = time.time()
    annoy_idx = AnnoyIndex(dim, annoy_metric)
    for i in range(n_data):
        annoy_idx.add_item(i, xb[i])
    annoy_idx.build(n_trees=10)
    annoy_build_time = time.time() - t0

    t0 = time.time()
    annoy_preds = [annoy_idx.get_nns_by_vector(v, k, search_k=1000) for v in xq]
    annoy_query_time = (time.time() - t0) * 1000 / nq  # ms/query
    annoy_recall = compute_recall(annoy_preds, gt_idx, k)
    results["ANNOY"] = f"{annoy_recall:.2f} ({annoy_query_time:.3f} ms)"

    # ----------------------------------------------------
    # B. FAISS (IndexIVFFlat)
    # ----------------------------------------------------
    t0 = time.time()
    nlist = int(np.sqrt(n_data)) if n_data > 1000 else 10
    quantizer = faiss.IndexFlatIP(dim) if metric == "cosine" else faiss.IndexFlatL2(dim)
    faiss_idx = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss_metric)
    faiss_idx.train(xb)
    faiss_idx.add(xb)
    faiss_build_time = time.time() - t0

    faiss_idx.nprobe = max(1, nlist // 10)
    t0 = time.time()
    _, faiss_preds = faiss_idx.search(xq, k)
    faiss_query_time = (time.time() - t0) * 1000 / nq
    faiss_recall = compute_recall(faiss_preds, gt_idx, k)
    results["FAISS"] = f"{faiss_recall:.2f} ({faiss_query_time:.3f} ms)"

    # ----------------------------------------------------
    # C. HNSW
    # ----------------------------------------------------
    t0 = time.time()
    hnsw_idx = hnswlib.Index(space=hnsw_metric, dim=dim)
    hnsw_idx.init_index(max_elements=n_data, ef_construction=100, M=16)
    hnsw_idx.add_items(xb)
    hnsw_build_time = time.time() - t0

    hnsw_idx.set_ef(50)
    t0 = time.time()
    hnsw_preds, _ = hnsw_idx.knn_query(xq, k=k)
    hnsw_query_time = (time.time() - t0) * 1000 / nq
    hnsw_recall = compute_recall(hnsw_preds, gt_idx, k)
    results["HNSW"] = f"{hnsw_recall:.2f} ({hnsw_query_time:.3f} ms)"

    return results

# Eksekusi seluruh skenario percobaan
scenarios = [
    (1000, 2, "euclidean", "1000/2D (Euclidean)"),
    (1000, 2, "cosine",    "1000/2D (Cosine)"),
    (1000000, 5, "euclidean", "1000000/5D (Euclidean)"),
    (1000000, 5, "cosine",    "1000000/5D (Cosine)")
]

table_rows = []
for n, d, m, label in scenarios:
    res = benchmark_scenario(n, d, metric=m)
    res["Jumlah Data/Dimensi"] = label
    table_rows.append(res)

df_result = pd.DataFrame(table_rows)[["Jumlah Data/Dimensi", "ANNOY", "FAISS", "HNSW"]]
print("\n" + df_result.to_markdown(index=False))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 9.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 46.9 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520033 sha256=f955eb77e3b2f589528397bf78ad26ea806da645eab9c60e0630404e156650c0
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=0bba1707c3398b51e37b35cf7b49f500434f7fa3828b6a4eecfe148a46753745
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

### **Hasil Pengujian & Analisis Perbandingan Algoritma ANN**

Berikut adalah ringkasan performansi dari pengujian yang telah dijalankan, menggunakan format **Recall@10 (Query Time per Query dalam milidetik)**.

#### **1. Tabel Hasil Pengujian**

| Jumlah Data/Dimensi | ANNOY | FAISS (IVFFlat) | HNSW |
| :--- | :--- | :--- | :--- |
| **1000/2D (Euclidean)** | 1.00 (0.133 ms) | 0.90 (0.003 ms) | 1.00 (0.012 ms) |
| **1000/2D (Cosine)** | 1.00 (0.114 ms) | 0.96 (0.003 ms) | 1.00 (0.009 ms) |
| **1000000/5D (Euclidean)** | 1.00 (0.298 ms) | 1.00 (0.689 ms) | 1.00 (0.035 ms) |
| **1000000/5D (Cosine)** | 1.00 (0.234 ms) | 1.00 (1.098 ms) | 1.00 (0.036 ms) |

---

#### **2. Analisis & Pembahasan**

*   **Pengaruh Skala Data ($1.000$ vs $1.000.000$ data):**
    *   Pada skala kecil ($1.000$ data), seluruh metode pencarian menghasilkan akurasi yang sangat tinggi (Recall mendekati atau mencapai $1.00$) karena ruang pencarian yang masih sempit.
    *   Pada skala besar ($1.000.000$ data), **HNSW** terbukti sangat unggul dalam kecepatan query (~$0.03$ ms) dengan tetap menjaga akurasi sempurna ($1.00$).
*   **Pengaruh Dimensi ($2D$ vs $5D$):**
    *   Peningkatan dimensi dari $2D$ ke $5D$ meningkatkan kompleksitas perhitungan jarak.
    *   **FAISS** mengalami peningkatan waktu pencarian yang cukup signifikan pada dimensi yang lebih tinggi dibandingkan dengan **HNSW** dan **ANNOY** yang relatif lebih stabil.
*   **Pengaruh Metrik Jarak (Euclidean vs Cosine):**
    *   Secara umum, pencarian berbasis metrik *Cosine* (setelah proses normalisasi vektor $L_2$) memiliki karakteristik kecepatan yang hampir setara dengan *Euclidean*.
*   **Kesimpulan Perbandingan Algoritma:**
    *   **HNSW (Hierarchical Navigable Small World):** Memberikan performansi terbaik dari sisi kombinasi akurasi (Recall) dan kecepatan pencarian (latensi terendah pada skala besar).
    *   **FAISS (IVFFlat):** Sangat efisien dalam penggunaan memori dan memiliki kecepatan latih (*train*) yang sangat baik, namun waktu pencariannya pada konfigurasi CPU default sedikit lebih lambat dibanding HNSW pada dataset besar.
    *   **ANNOY (Approximate Nearest Neighbors Oh Yeah):** Memiliki performansi pencarian yang konsisten dan seimbang untuk data skala menengah, namun latensinya sedikit di bawah HNSW pada beban kerja pencarian tinggi.